[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Observabilidad, *smoke tests* y *rollback*.

Este curso trata la observabilidad como parte del sistema de entrega y no como una tarea posterior del equipo de operaciones. Rollback se estudia como respuesta guiada por señales, no como improvisacion.

## Objetivos.

- Diseñar verificaciones posteriores al *deploy*.
- Distinguir *logs*, metricas, trazas y alertas como señales complementarias.
- Relacionar *smoke tests* con liberacion segura.
- Entender *rollback* como capacidad y no como improvisacion.

## Verificar antes de dar por bueno el despliegue.

Un *smoke test* comprueba que el servicio recien desplegado responde en su ruta minima esperada. Esa verificacion debe ir acompañada de observabilidad basica: *logs*, metricas, estado del proceso, latencia, tasa de errores y señales de saturacion.

La idea central es simple: si el *pipeline* despliega pero nadie puede observar el estado real del servicio, todavia no existe entrega defendible. Rollback no es solo volver a una version anterior; es conservar trazabilidad suficiente para saber a que version volver, bajo que criterio hacerlo y como validar que la recuperacion funciono.

## Pila minima de observabilidad.

En un curso de *CI/CD* no hace falta cubrir una plataforma completa de observabilidad, pero si es necesario fijar el modelo operativo minimo:

- *logs* estructurados para reconstruir eventos y errores;
- metricas de salud como latencia, throughput y error rate;
- trazas cuando un request atraviesa varios componentes;
- alertas que traduzcan desviaciones tecnicas en accion operativa.

Una entrega continua madura define desde antes del despliegue que señales se observaran, con que umbrales y quien responde cuando algo sale del rango esperado.

## SLI, SLO y alertas.

Los *smoke tests* sirven para responder 'arranco o no arranco', pero la observabilidad responde 'esta funcionando dentro de parametros aceptables'. Por eso conviene introducir al menos estas ideas:

- SLI: una medicion observable, por ejemplo tasa de error o latencia p95;
- SLO: el objetivo esperado para ese indicador;
- alerta: la regla que dispara investigacion o *rollback* cuando el servicio se aparta del objetivo.

## Aplicacion a las rutas del curso.

La observabilidad no se revisa igual en todos los destinos de despliegue. En este curso conviene contrastar las dos rutas principales:

- Google *Cloud Functions*: revisar invocaciones, errores, duracion, *logs* estructurados y comportamiento por version desplegada;
- *Linux* por SSH: revisar *logs* de la aplicacion, proxy inverso, procesos o contenedores, consumo de CPU y memoria, puertos expuestos y estado del servicio.

Eso permite mostrar que el mismo *pipeline* termina en entornos distintos, pero ambos necesitan evidencia operativa despues del *deploy*.

## Runbook minimo posterior al despliegue.

Un runbook breve pero util deberia responder al menos estas preguntas:

- que panel, comando o fuente de *logs* se consulta primero;
- que umbral obliga a investigar y cual obliga a revertir;
- quien aprueba el *rollback* en `prod`;
- como se valida que la version recuperada quedo sana.

Sin ese runbook, la observabilidad queda como telemetria pasiva y no como capacidad real de operacion.

In [ ]:
import urllib.request
import json


def smoke_test(url: str, timeout: int = 10) -> bool:
    """Verifica que el servicio responde HTTP 200 en la ruta mínima."""
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            status = resp.status
            print(f"  status : {status}")
            ctype = resp.headers.get("Content-Type", "")
            if "application/json" in ctype:
                payload = json.loads(resp.read())
                print(f"  body   : {payload}")
            return status == 200
    except Exception as exc:
        print(f"  error  : {exc}")
        return False


# Uso real (reemplaza SERVICE_URL con la URL del despliegue):
# SERVICE_URL = "https://<region>-<project>.cloudfunctions.net/<function>/alumnos"
# assert smoke_test(SERVICE_URL), "Smoke test fallido — activar rollback"

# Dry-run ejecutable (httpbin.org es un servicio de prueba público):
ok = smoke_test("https://httpbin.org/status/200")
print("smoke test pasó:", ok)


## Smoke test como paso de *GitHub Actions*.

```yaml
- name: Smoke test post-despliegue
  run: |
    STATUS=$(curl -s -o /dev/null -w "%{http_code}" "${{ env.SERVICE_URL }}/alumnos")
    if [ "$STATUS" != "200" ]; then
      echo "Smoke test fallido (HTTP $STATUS) — revirtiendo revisión"
      gcloud run services update-traffic ${{ env.SERVICE_NAME }} \
        --to-revisions=PREVIOUS=100 \
        --region=${{ env.REGION }}
      exit 1
    fi
    echo "Smoke test OK"
```

El *rollback* revierte el tráfico a la revisión anterior de *Cloud Run* sin intervención manual. Si la revisión anterior también falla, el equipo recibe la alerta y decide el paso siguiente. Este patrón cierra el ciclo de despliegue: no basta con que el *pipeline* termine verde; el servicio tiene que responder en producción.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>